<a href="https://colab.research.google.com/github/qwe948714-beep/113408012_CountingFail/blob/main/p%E8%AA%9E%E8%A8%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install requests beautifulsoup4 pypdf reportlab -q
print("✅ 套件完成")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.6 MB/s eta 0:00:00
✅ 套件完成


In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote
from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from io import BytesIO
from google.colab import files

pdfmetrics.registerFont(UnicodeCIDFont("HeiseiMin-W3"))

print("✅ 模組完成")

✅ 模組完成


In [6]:
def search_web(keyword, max_results=20):
    url = f"https://html.duckduckgo.com/html/?q={quote(keyword)}"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        results = []

        for block in soup.select(".result"):
            title = block.select_one(".result__title a")
            if title:
                results.append({
                    "title": title.get_text(strip=True),
                    "link": title.get("href")
                })

            if len(results) >= max_results:
                break

        return results
    except:
        return []

In [7]:

job_title = input("職缺：")
location = input("地區：")
salary_min = int(input("最低薪資："))

raw = search_web(f"{job_title} 徵才 {location}", 30)

jobs_list = []
for r in raw:
    score = 0

    if job_title in r["title"]:
        score += 3
    if location in r["title"]:
        score += 2

    jobs_list.append({
        "title": r["title"],
        "company": "網路來源",
        "link": r["link"],
        "score": score
    })

# 排序
jobs_list.sort(key=lambda x: x["score"], reverse=True)

tracked_jobs = jobs_list  # ⭐ 全部推薦（你要求不要限制）

print("\n📌 AI推薦職缺（已排序）")
for j in jobs_list[:10]:
    print(j["title"])

職缺：會計
地區：台北
最低薪資：50000

📌 AI推薦職缺（已排序）
「會計」台北市最新找工作職缺｜2026年5月－104人力銀行
#徵才 #徵才 台北市大同區 會計助理/專員 - 工作板 | Dcard
「會計」找工作查詢｜2026年5月-yes123求職網
「會計」職缺 - 2026年5月熱門工作機會｜1111人力銀行
200 個「財務會計」的職缺（台灣） （18 個最新）
會計工作，會計職缺, 91 個職位 - Jooble
在台灣的會計人員工作 | Careerjet
勤業眾信聯合會計師事務所 | Deloitte Taiwan | 審計、管理顧問、財務諮詢、風險管理、稅務服務
超過 900 份會計人員職缺，2026年5月7日的就業機會 | Indeed
臺北市政府機關學校求職徵才網


In [8]:
personal_info = {
    "name": input("姓名："),
    "phone": input("電話："),
    "email": input("Email："),
    "city": location,
    "experience": input("工作經驗：")
}

edu_info = {
    "school": input("學校："),
    "major": input("科系："),
    "skills": input("技能（逗號）：").split(",")
}

姓名：黃紫雲
電話：0988888888
Email：qwe
工作經驗：無
學校：中央大學
科系：財金系
技能（逗號）：沒


In [9]:
import random

def generate_ai_intro():
    templates = [
        f"我目前就讀於{edu_info['school']}，主修{edu_info['major']}。對於{job_title}相關工作充滿興趣，並持續累積實務能力。",
        f"在學期間我主要專注於{edu_info['major']}領域，並希望未來能投入{job_title}相關產業發展。",
        f"我具備基礎專業能力，並積極尋找{job_title}的實務機會，希望累積更多產業經驗。"
    ]

    base = random.choice(templates)
    skills = "、".join(edu_info["skills"])

    extra = f"\n目前具備技能包含：{skills}。" if skills else ""

    exp = f"\n過去經驗：{personal_info['experience']}" if personal_info["experience"] else "\n目前尚無正式工作經驗"

    return base + extra + exp

ai_intro = generate_ai_intro()

In [10]:
def make_resume_pdf():
    packet = BytesIO()
    can = canvas.Canvas(packet)

    can.setFont("HeiseiMin-W3", 12)

    # =====================
    # 左上角照片框
    # =====================
    can.rect(40, 700, 100, 120)
    can.drawString(45, 760, "PHOTO")

    # =====================
    # 基本資料
    # =====================
    can.drawString(160, 780, f"姓名：{personal_info['name']}")
    can.drawString(160, 760, f"職缺：{job_title}")
    can.drawString(160, 740, f"Email：{personal_info['email']}")
    can.drawString(160, 720, f"城市：{personal_info['city']}")

    # =====================
    # 自介
    # =====================
    y = 680
    can.drawString(40, y, "自我介紹")
    y -= 20

    for line in ai_intro.split("\n"):
        can.drawString(40, y, line)
        y -= 18

    # =====================
    # 技能
    # =====================
    y -= 10
    can.drawString(40, y, "技能")
    y -= 20

    can.drawString(40, y, "、".join(edu_info["skills"]))

    can.save()

    packet.seek(0)

    overlay = PdfReader(packet)
    base = PdfWriter()

    writer = PdfWriter()

    page = overlay.pages[0]
    writer.add_page(page)

    output = "resume.pdf"

    with open(output, "wb") as f:
        writer.write(f)

    return output

resume_file = make_resume_pdf()
files.download(resume_file)
print("✅ 履歷完成")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 履歷完成


In [14]:
def make_resume_pdf():

    packet = BytesIO()

    can = canvas.Canvas(packet)

    can.setFont("HeiseiMin-W3", 12)

    # =====================
    # 姓名
    # =====================

    can.setFillColorRGB(1,1,1)
    can.rect(120, 700, 250, 20, fill=1)

    can.setFillColorRGB(0,0,0)c
    can.drawString(125, 705, personal_info["name"])

    # =====================
    # 電話
    # =====================

    can.setFillColorRGB(1,1,1)
    can.rect(120, 650, 250, 20, fill=1)

    can.setFillColorRGB(0,0,0)
    can.drawString(125, 655, personal_info["phone"])

    # =====================
    # Email
    # =====================

    can.setFillColorRGB(1,1,1)
    can.rect(120, 620, 250, 20, fill=1)

    can.setFillColorRGB(0,0,0)
    can.drawString(125, 625, personal_info["email"])

    can.save()

    packet.seek(0)

    overlay = PdfReader(packet)

    template = PdfReader("Green Aesthetic Creative Cv Resume.pdf")

    page = template.pages[0]

    page.merge_page(overlay.pages[0])

    writer = PdfWriter()

    writer.add_page(page)

    output = "filled_resume.pdf"

    with open(output, "wb") as f:
        writer.write(f)

    return output